In [ ]:
import re

import numpy as np
import pandas as pd
from scipy.stats import chi2
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

In [ ]:
# Path to the token-level corpus data
PATH = "all_words_from_corpus.csv" # all tokens from corpus

# Gender labels used in the dataset
FEMALE = 0
MALE = 1


def extract_lemma(morph, word=None):
    """
    Extract the lemma from a morphological annotation.
    Falls back to the original token if the lemma is unavailable.
    """
    if pd.isna(morph):
        return str(word).lower() if word is not None else None

    morph = str(morph).strip()
    match = re.search(r"^\s*([^+\s]+)\+", morph)

    if match:
        return match.group(1).lower()

    return str(word).lower() if word is not None else None


def extract_pos(morph):
    """
    Extract the POS tag from a morphological annotation.
    """
    if pd.isna(morph):
        return None

    match = re.search(r"//_([^_]+)_", str(morph))
    return match.group(1) if match else None


# Load the corpus
df = pd.read_csv(PATH, keep_default_na=False)

# Extract lemmas and POS tags
df["lemma"] = [
    extract_lemma(morph, word)
    for morph, word in zip(df["morph"], df["word"])
]
df["pos"] = df["morph"].apply(extract_pos)

# Keep only rows with complete metadata
df = df.dropna(subset=["lemma", "ID", "gender"])

# Normalize lemma strings
df["lemma"] = df["lemma"].str.lower().str.strip()

# Remove empty and numeric-only tokens
df = df[df["lemma"] != ""]
df = df[~df["lemma"].str.fullmatch(r"\d+")]

In [ ]:
# Minimum inclusion criteria for candidate lemmas
MIN_TOTAL = 20
MIN_SPEAKERS = 5


In [ ]:

# Use the full corpus as the frequency denominator
reference = pd.read_csv(
    PATH,
    usecols=["ID", "gender", "word"],
    keep_default_na=False
)

reference = reference.dropna(subset=["ID", "gender", "word"])
reference = reference[
    reference["gender"].isin([FEMALE, MALE])
    & reference["word"].astype(str).str.strip().ne("")
]

total_female = (reference["gender"] == FEMALE).sum()
total_male = (reference["gender"] == MALE).sum()

# Style-focused POS categories, extended with two target lemmas from
# the manuscript's particle categories, regardless of morphological POS.
STYLE_POS = {"P", "D", "I", "J"}
TARGET_LEMMAS = {"jumal", "kurat"}

style = df[
    (df["pos"].isin(STYLE_POS) | df["lemma"].isin(TARGET_LEMMAS))
    & df["gender"].isin([FEMALE, MALE])
].copy()

# Apply minimum frequency and speaker-coverage thresholds
lemma_stats = style.groupby("lemma").agg(
    total_tokens=("lemma", "size"),
    n_speakers=("ID", "nunique")
)

valid_lemmas = lemma_stats[
    (lemma_stats["total_tokens"] >= MIN_TOTAL)
    & (lemma_stats["n_speakers"] >= MIN_SPEAKERS)
].index

style_key = style[style["lemma"].isin(valid_lemmas)].copy()

# Calculate keyness against the full corpus denominators
results = []

for lemma, sub in style_key.groupby("lemma"):

    female_count = int((sub["gender"] == FEMALE).sum())
    male_count = int((sub["gender"] == MALE).sum())

    contingency = np.array([
        [female_count, total_female - female_count],
        [male_count, total_male - male_count]
    ])

    # Log-likelihood ratio (G-test) without Yates correction
    G2, p, _, _ = chi2_contingency(
        contingency,
        correction=False,
        lambda_="log-likelihood"
    )

    female_per_10k = female_count / total_female * 10000
    male_per_10k = male_count / total_male * 10000

    # Positive values indicate higher relative frequency among women
    log_ratio = np.log2(
        ((female_count + 0.5) / total_female)
        / ((male_count + 0.5) / total_male)
    )

    results.append({
        "lemma": lemma,
        "female_count": female_count,
        "male_count": male_count,
        "female_per_10k": female_per_10k,
        "male_per_10k": male_per_10k,
        "log_ratio_female_vs_male": log_ratio,
        "G2": G2,
        "p": p
    })

keyness_style = pd.DataFrame(results)

# Apply Benjamini-Hochberg correction across all tested lemmas
keyness_style["FDR"] = multipletests(
    keyness_style["p"],
    method="fdr_bh"
)[1]

keyness_style["more_frequent_in"] = np.select(
    [
        keyness_style["log_ratio_female_vs_male"] > 0,
        keyness_style["log_ratio_female_vs_male"] < 0
    ],
    ["female", "male"],
    default="equal"
)

keyness_style["abs_log_ratio"] = (
    keyness_style["log_ratio_female_vs_male"].abs()
)

# Document why each lemma qualified for the expanded POS-based selection.
keyness_style["selection_basis"] = np.where(
    keyness_style["lemma"].isin(TARGET_LEMMAS),
    "target_lemma_all_POS", "POS_P_D_I_J"
)

# Select significant lemmas
significant = keyness_style[
    keyness_style["FDR"] < 0.05
].copy()

# Select the top 15 lemmas by absolute log ratio for each gender
top_female = (
    significant[significant["more_frequent_in"] == "female"]
    .sort_values("abs_log_ratio", ascending=False)
    .head(15)
)

top_male = (
    significant[significant["more_frequent_in"] == "male"]
    .sort_values("abs_log_ratio", ascending=False)
    .head(15)
)

# Export complete results and publication tables
keyness_style.to_csv(
    "keyness_style_lemmas_expanded_full_denominator.csv",
    index=False
)

top_female.to_csv(
    "keyness_top15_female_expanded_full_denominator.csv",
    index=False
)

top_male.to_csv(
    "keyness_top15_male_expanded_full_denominator.csv",
    index=False
)

# Report denominators and selection details
print("Female corpus tokens:", total_female)
print("Male corpus tokens:", total_male)
print("Total corpus tokens:", total_female + total_male)
print("Tested lemmas:", len(keyness_style))
print("FDR-significant lemmas:", len(significant))
print("Top female lemmas:", len(top_female))
print("Top male lemmas:", len(top_male))
print("Jumal counts:", keyness_style.loc[keyness_style["lemma"].eq("jumal"), ["female_count", "male_count"]].to_dict("records"))
print("Kurat counts:", keyness_style.loc[keyness_style["lemma"].eq("kurat"), ["female_count", "male_count"]].to_dict("records"))